### Import the required libraries

Import the PostgreSQL driver and Kafka producer, plus `json` to serialize messages and `time` to control the polling interval.

In [1]:
import psycopg2
from kafka import KafkaProducer
import json
import time

### Connect to PostgreSQL

Open a connection to the local `streaming_db` database so the following cells can read cryptocurrency prices from `crypto_price`.

In [2]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="streaming_db",
    user="postgres",
    password=""
)

### Send a sample Kafka message

Create a producer connected to Kafka at `localhost:9092`, converting message dictionaries into UTF-8 JSON bytes. Send a sample BTC price of 30000 to the `crypto_stream` topic, then call `flush()` to wait for pending sends to complete before printing a confirmation.

In [3]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

producer.send("crypto_stream", {"coin": "BTC", "price": 30000})
producer.flush()
print("✅ berhasil kirim")

✅ berhasil kirim


### Publish database rows to Kafka continuously

Create a producer and database cursor, then start tracking rows with `last_id = 0`. Query `crypto_price` for IDs greater than the last processed ID in ascending order, and publish each row to `crypto_stream` with its coin, price, and timestamp. Flush pending messages and pause for 3 seconds between polls. Existing rows with positive IDs are included on the first pass; rerunning this cell resets the in-memory ID and can send them again. The loop runs until interrupted or an unhandled error occurs.

In [4]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

cur = conn.cursor()
last_id = 0

while True:
    cur.execute(
        "SELECT * FROM crypto_price WHERE id > %s ORDER BY id",
        (last_id,)
    )
    rows = cur.fetchall()

    for r in rows:
        last_id = r[0]
        producer.send("crypto_stream", {
            "coin": r[1],
            "price": r[2],
            "time": str(r[3])
        })

    producer.flush()
    time.sleep(3)

KeyboardInterrupt: 